# Phase 1: Dataset Understanding, Preprocessing, EDA & Random Forest Baseline

**Project Title:** Explainable AI-Based Air Pollution Prediction and Source Attribution  
**Repository:** https://github.com/BhanuDanda-09/Air-Pollution-XAI.git  
**Phase:** Phase 1 — Dataset Analysis, Data Preprocessing, Exploratory Data Analysis, and Random Forest Baseline Model.

---
### Objectives of this Notebook:
1. **Dataset Inspection:** Load `Delhi Final(1).xlsx` and inspect dimensions, column types, missingness, duplicates, and physical validity.
2. **Data Preprocessing:** Clean extra columns, handle duplicates with scientific justification, construct an imputation pipeline for missing Solar Radiation (`SR`), and perform a leak-free 80/20 train-test split (`random_state=42`).
3. **Exploratory Data Analysis (EDA):** Generate publication-quality diagnostic plots analyzing PM2.5 distributions, correlations, and meteorological feature relationships.
4. **Baseline Modeling:** Train a `RandomForestRegressor` baseline pipeline on the 5 confirmed input features (`RH`, `WD`, `SR`, `BP`, `AT`).
5. **Model Evaluation:** Measure actual test-set performance metrics (MAE, MSE, RMSE, R²), plot actual vs. predicted values, residuals, and feature importances.
6. **Inference Test:** Validate inference on an unseen test sample.


In [ ]:
import os
import sys
import json
import joblib
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Add project root to sys.path
sys.path.insert(0, os.path.abspath('..'))

from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Set plot styling
sns.set_theme(style='whitegrid', font='sans-serif')
plt.rcParams['figure.dpi'] = 150
print('Libraries imported successfully.')


## 1. Dataset Loading & Inspection
We load the raw Excel dataset `Delhi Final(1).xlsx` and examine its structure.


In [ ]:
dataset_path = '../dataset/Delhi Final(1).xlsx'
raw_df = pd.read_excel(dataset_path)

print('Dataset Shape:', raw_df.shape)
print('\nAll Columns:', raw_df.columns.tolist())
raw_df.head()


In [ ]:
# Check core features vs artifact columns
core_cols = ['PM2.5', 'RH', 'WD', 'SR', 'BP', 'AT']
core_df = raw_df[core_cols]

print('--- Missing Values in Core Variables ---')
print(core_df.isnull().sum())

print('\n--- Statistical Summary ---')
display(core_df.describe().T)

print('\nExact duplicate rows in core features:', core_df.duplicated().sum())


## 2. Data Preprocessing & Train-Test Split

### Scientific Decisions:
1. **Artifact Removal:** Dropped columns `Unnamed: 6` to `Unnamed: 12` which are empty sheet artifacts.
2. **Duplicate Handling:** Dropped 156 exact duplicate rows to prevent train-test data leakage across random splits.
3. **Train-Test Split:** 80% Training (1,621 samples) and 20% Testing (406 samples) with `random_state=42`.
4. **Missing Value Imputation:** Only `SR` has 12 missing values (0.55%). We use `SimpleImputer(strategy='median')` fitted exclusively on the training set.


In [ ]:
# Deduplicate
cleaned_df = core_df.drop_duplicates().reset_index(drop=True)
print('Cleaned Dataset Shape:', cleaned_df.shape)

# Separate X and y
features = ['RH', 'WD', 'SR', 'BP', 'AT']
target = 'PM2.5'

X = cleaned_df[features]
y = cleaned_df[target]

# Train-test split (80% train, 20% test, seed=42)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, shuffle=True
)

print(f'Training Set: {X_train.shape[0]} samples')
print(f'Testing Set:  {X_test.shape[0]} samples')


## 3. Exploratory Data Analysis (EDA)

We visualize the distribution of PM2.5, pairwise Pearson correlations, and scatter trends against meteorological variables.


In [ ]:
# Target distribution & boxplot
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram
sns.histplot(cleaned_df['PM2.5'], kde=True, color='#d95f02', bins=35, ax=axes[0], edgecolor='k')
pm_mean = cleaned_df['PM2.5'].mean()
pm_median = cleaned_df['PM2.5'].median()
axes[0].axvline(pm_mean, color='red', linestyle='--', label=f'Mean: {pm_mean:.1f}')
axes[0].axvline(pm_median, color='blue', linestyle='-', label=f'Median: {pm_median:.1f}')
axes[0].set_title('PM2.5 Concentration Distribution', fontweight='bold')
axes[0].set_xlabel('PM2.5 (µg/m³)')
axes[0].legend()

# Boxplot
sns.boxplot(x=cleaned_df['PM2.5'], color='#fc8d62', ax=axes[1])
axes[1].set_title('PM2.5 Concentration Boxplot', fontweight='bold')
axes[1].set_xlabel('PM2.5 (µg/m³)')

plt.tight_layout()
plt.show()


In [ ]:
# Pearson correlation heatmap
plt.figure(figsize=(8, 6))
corr = cleaned_df.corr()
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', vmin=-1, vmax=1, center=0, square=True, linewidths=1.0)
plt.title('Pearson Correlation Heatmap', fontweight='bold', fontsize=14)
plt.tight_layout()
plt.show()


In [ ]:
# Meteorological scatter plots against PM2.5
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.flatten()

scatter_configs = [
    ('RH', 'Relative Humidity (%)', '#1f78b4'),
    ('WD', 'Wind Direction (°)', '#33a02c'),
    ('SR', 'Solar Radiation (W/m²)', '#e31a1c'),
    ('BP', 'Barometric Pressure (mmHg)', '#ff7f00'),
    ('AT', 'Air Temperature (°C)', '#6a3d9a'),
]

for idx, (col, lbl, clr) in enumerate(scatter_configs):
    ax = axes[idx]
    sns.regplot(data=cleaned_df, x=col, y='PM2.5', scatter_kws={'alpha': 0.35, 'color': clr, 's': 20},
                line_kws={'color': 'black', 'linewidth': 1.8}, ax=ax)
    r_val = corr.loc[col, 'PM2.5']
    ax.set_title(f'{lbl} vs PM2.5 (r = {r_val:+.2f})', fontweight='bold')
    ax.set_xlabel(lbl)
    ax.set_ylabel('PM2.5 (µg/m³)')

fig.delaxes(axes[5])
plt.suptitle('Meteorological Variables vs. PM2.5 Concentration', fontsize=15, fontweight='bold', y=0.98)
plt.tight_layout()
plt.show()


## 4. Random Forest Baseline Model Training

We assemble a unified Scikit-learn `Pipeline` that:
1. Imputes missing values with the training-derived median.
2. Trains a `RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)`.


In [ ]:
rf_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('regressor', RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1))
])

# Fit strictly on training set
rf_pipeline.fit(X_train, y_train)

# Save model bundle
os.makedirs('../models', exist_ok=True)
joblib.dump(rf_pipeline, '../models/random_forest.pkl')
print('Random Forest Baseline Model trained and saved to ../models/random_forest.pkl')


## 5. Model Evaluation on Held-out Test Data (20%)

We calculate MAE, MSE, RMSE, and R² on the unseen test set (406 observations).


In [ ]:
# Predict on test set
y_pred_test = rf_pipeline.predict(X_test)
y_pred_train = rf_pipeline.predict(X_train)

# Calculate metrics
test_mae = mean_absolute_error(y_test, y_pred_test)
test_mse = mean_squared_error(y_test, y_pred_test)
test_rmse = np.sqrt(test_mse)
test_r2 = r2_score(y_test, y_pred_test)

train_mae = mean_absolute_error(y_train, y_pred_train)
train_rmse = np.sqrt(mean_squared_error(y_train, y_pred_train))
train_r2 = r2_score(y_train, y_pred_train)

metrics_df = pd.DataFrame([
    {'Metric': 'MAE (µg/m³)', 'Training Set': f'{train_mae:.2f}', 'Testing Set': f'{test_mae:.2f}'},
    {'Metric': 'MSE (µg/m²)²', 'Training Set': f'{mean_squared_error(y_train, y_pred_train):.2f}', 'Testing Set': f'{test_mse:.2f}'},
    {'Metric': 'RMSE (µg/m³)', 'Training Set': f'{train_rmse:.2f}', 'Testing Set': f'{test_rmse:.2f}'},
    {'Metric': 'R² Score', 'Training Set': f'{train_r2:.4f}', 'Testing Set': f'{test_r2:.4f}'},
])
display(metrics_df)


In [ ]:
# Evaluation Visualizations
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 1. Actual vs Predicted
axes[0].scatter(y_test, y_pred_test, color='#1f77b4', alpha=0.55, edgecolor='k', s=30)
min_v = min(y_test.min(), y_pred_test.min()) - 10
max_v = max(y_test.max(), y_pred_test.max()) + 10
axes[0].plot([min_v, max_v], [min_v, max_v], 'r--', label='1:1 Perfect Fit')
axes[0].set_xlim(min_v, max_v)
axes[0].set_ylim(min_v, max_v)
axes[0].set_title(f'Actual vs. Predicted (R² = {test_r2:.3f})', fontweight='bold')
axes[0].set_xlabel('Actual PM2.5 (µg/m³)')
axes[0].set_ylabel('Predicted PM2.5 (µg/m³)')
axes[0].legend()

# 2. Residual Plot
residuals = y_test - y_pred_test
axes[1].scatter(y_pred_test, residuals, color='#2ca02c', alpha=0.55, edgecolor='k', s=30)
axes[1].axhline(0, color='red', linestyle='--', label='Zero Error Line')
axes[1].set_title(f'Residual Plot (RMSE = {test_rmse:.2f} µg/m³)', fontweight='bold')
axes[1].set_xlabel('Predicted PM2.5 (µg/m³)')
axes[1].set_ylabel('Residual (Actual - Predicted)')
axes[1].legend()

# 3. Feature Importance
rf = rf_pipeline.named_steps['regressor']
imp_df = pd.DataFrame({'Feature': features, 'Importance': rf.feature_importances_}).sort_values('Importance', ascending=True)
axes[2].barh(imp_df['Feature'], imp_df['Importance'], color='#3182bd', edgecolor='k')
for bar in axes[2].patches:
    axes[2].text(bar.get_width() + 0.005, bar.get_y() + bar.get_height() / 2, f'{bar.get_width()*100:.1f}%', va='center', fontweight='bold')
axes[2].set_title('Random Forest Feature Importance', fontweight='bold')
axes[2].set_xlabel('Gini Impurity Decrease')

plt.tight_layout()
plt.show()


## 6. Single Sample Inference Test

We verify that the saved pipeline can be loaded and used to predict PM2.5 on a single observation.


In [ ]:
# Load model from disk
loaded_model = joblib.load('../models/random_forest.pkl')

# Pick an example from X_test
sample_x = X_test.iloc[[0]]
actual_y = y_test.iloc[0]
pred_y = loaded_model.predict(sample_x)[0]

print('Input Sample Features:')
display(sample_x)
print(f'Actual PM2.5:    {actual_y:.2f} µg/m³')
print(f'Predicted PM2.5: {pred_y:.2f} µg/m³')
print(f'Absolute Error:  {abs(actual_y - pred_y):.2f} µg/m³')


## 7. Phase 1 Scientific Scope & Limitations

1. **No Timestamp Column:** The dataset does not contain explicit timestamp headers. Therefore, observations are treated as cross-sectional tabular samples. It is not suitable for genuine autoregressive time-series forecasting (e.g. ARIMA, LSTM) without genuine sequential timestamps.
2. **Feature Domain:** Only PM2.5 and 5 meteorological variables (`RH`, `WD`, `SR`, `BP`, `AT`) are present. Additional pollutants (NO₂, SO₂, CO, O₃) were not fabricated.
3. **Feature Importance vs. Causation:** Feature importance demonstrates model variance reduction, not physical proof of emission sources. Phase 1 provides a PM2.5 baseline prediction model, not a complete source attribution system.
4. **Scope:** Advanced models (XGBoost, LSTM), Explainability frameworks (SHAP, LIME), full-stack UI, and cloud deployment are reserved for subsequent phases.
